# 🧠 Support Ticket Classification Model  
### Category & Priority Prediction using Local AI-Labeled Data

---

## 📌 Purpose of This Notebook

This notebook trains and evaluates a **machine learning–based support ticket classifier** using the **latest locally categorized and prioritized dataset**.

The model is designed to:
- Predict the **ticket category** (e.g., infrastructure, billing, security, hardware)
- Assign a **business-aligned priority** (High / Medium / Low)
- Handle real-world edge cases such as:


In [60]:
import torch

import pandas as pd 
import sklearn 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report


torch.__version__

'2.10.0+cu126'

In [61]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Using pandas, Let's read the data

In [62]:
data = pd.read_csv('classified_tickets.csv')
data.head()

,id,description,category,priority
0,1,"Account Disruption Dear Customer Support Team,...",account,High
1,2,Query About Smart Home System Integration Feat...,other,Low
2,3,Inquiry Regarding Invoice Details Dear Custome...,billing,Low
3,4,Question About Marketing Agency Software Compa...,other,Low
4,5,"Feature Query Dear Customer Support,\n\nI hope...",other,Low


### Splitting the data to train-test split

In [63]:
train_df , test_df = train_test_split(data,
                                    test_size=0.2,
                                    random_state=42)
train_df.head()

,id,description,category,priority
12909,26539,Support Request for Safeguarding Medical Data ...,security,Low
8373,16526,Business Growth Assistance Looking to enhance ...,other,Low
23108,58735,Hospital Data Security Incident Report Custome...,security,High
4678,9438,Request for Updated Billing Information We are...,billing,Low
20810,54362,Problem with Signal Integration Dear Customer ...,integration,Low


### Initializing TF-IDF Vectorizer

In [64]:
tfidf = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    max_features=20000
)

### Initializing Priority and category encoder

In [65]:
priority_encoder = LabelEncoder()

y_train_priority = priority_encoder.fit_transform(train_df["priority"])
y_test_priority = priority_encoder.transform(test_df["priority"])


In [66]:
category_encoder = LabelEncoder()

y_train_category = category_encoder.fit_transform(train_df["category"])
y_test_category = category_encoder.transform(test_df["category"])
y_train_category[:10], y_test_category[:10]

(array([9, 8, 9, 1, 6, 9, 8, 8, 9, 8]), array([5, 9, 9, 9, 6, 8, 8, 6, 9, 8]))

In [67]:
X_train = tfidf.fit_transform(train_df["description"])
X_test = tfidf.transform(test_df["description"])
X_train.shape, X_test.shape, y_train_priority.shape, y_test_priority.shape, y_train_category.shape, y_test_category.shape


((19696, 20000), (4925, 20000), (19696,), (4925,), (19696,), (4925,))

In [68]:
priority_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"  # VERY important for priority
)
priority_model.fit(X_train, y_train_priority)
priority_preds = priority_model.predict(X_test)
print(classification_report(y_test_priority, priority_preds))

              precision    recall  f1-score   support

           0       0.89      0.86      0.88      1475
           1       0.96      0.89      0.92      2187
           2       0.78      0.90      0.84      1263

    accuracy                           0.89      4925
   macro avg       0.88      0.89      0.88      4925
weighted avg       0.89      0.89      0.89      4925



In [69]:
category_model = LinearSVC()

category_model.fit(X_train, y_train_category)
category_preds = category_model.predict(X_test)
print(classification_report(y_test_category, category_preds))


              precision    recall  f1-score   support

           0       0.73      0.51      0.60        72
           1       0.94      0.93      0.94       233
           2       0.79      0.81      0.80       299
           3       0.86      0.75      0.80         8
           4       0.89      0.53      0.67        30
           5       0.85      0.91      0.88       559
           6       0.88      0.89      0.88      1196
           7       0.87      0.73      0.79       100
           8       0.91      0.90      0.91      1297
           9       0.96      0.96      0.96      1131

    accuracy                           0.90      4925
   macro avg       0.87      0.79      0.82      4925
weighted avg       0.90      0.90      0.90      4925



In [70]:
import random
import numpy as np

def test_single_ticket(model, vectorizer, text, label_enc_cat, label_enc_pri):
    # Vectorize
    X = vectorizer.transform([text])

    # Predict
    cat_pred = model['category'].predict(X)[0]
    pri_pred = model['priority'].predict(X)[0]

    # Decode
    category = label_enc_cat.inverse_transform([cat_pred])[0]
    priority = label_enc_pri.inverse_transform([pri_pred])[0]

    return category, priority


In [71]:
text = """
Office applications fail to open after macOS update.
"""

category, priority = test_single_ticket(
    {
        'category': category_model,
        'priority': priority_model
    },
    tfidf,
    text,
    category_encoder,
    priority_encoder
)
ACCESS_BLOCK_KEYWORDS = {
    "fail to open",
    "cannot open",
    "unable to open",
    "won't open",
    "does not open",
    "blocked",
    "cannot access"
}
if any(k in text for k in ACCESS_BLOCK_KEYWORDS):
    if priority == "Low":
        priority = "Medium"

print("CATEGORY:", category)
print("PRIORITY:", priority)


CATEGORY: integration
PRIORITY: Medium


In [72]:
from sklearn.pipeline import Pipeline

priority_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("model", priority_model)
])

In [73]:
category_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("model", category_model)
])

In [ ]:
"""import joblib
import os

os.makedirs("backend/model", exist_ok=True)

joblib.dump(priority_pipeline, "backend/model/priority_model.joblib")
joblib.dump(category_pipeline, "backend/model/category_model.joblib")"""

['backend/model/category_model.joblib']

In [75]:
test_priority = priority_pipeline.predict(["VPN not working"])
test_category = category_pipeline.predict(["VPN not working"])

category = category_encoder.inverse_transform([test_category])[0]
priority = priority_encoder.inverse_transform([test_priority])[0]

print(priority, category)


Low integration


c:\CODING\MINI PROJECT - S6\venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\CODING\MINI PROJECT - S6\venv\Lib\site-packages\sklearn\preprocessing\_label.py:161: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [76]:
"""class ModelWithLabelDecoder:
    def __init__(self, pipeline, label_encoder):
        self.pipeline = pipeline
        self.label_encoder = label_encoder

    def predict(self, texts):
        encoded_preds = self.pipeline.predict(texts)
        return self.label_encoder.inverse_transform(encoded_preds)"""


'class ModelWithLabelDecoder:\n    def __init__(self, pipeline, label_encoder):\n        self.pipeline = pipeline\n        self.label_encoder = label_encoder\n\n    def predict(self, texts):\n        encoded_preds = self.pipeline.predict(texts)\n        return self.label_encoder.inverse_transform(encoded_preds)'

In [77]:
"""priority_model_final = ModelWithLabelDecoder(
    priority_pipeline,
    priority_encoder
)

category_model_final = ModelWithLabelDecoder(
    category_pipeline,
    category_encoder
)"""


'priority_model_final = ModelWithLabelDecoder(\n    priority_pipeline,\n    priority_encoder\n)\n\ncategory_model_final = ModelWithLabelDecoder(\n    category_pipeline,\n    category_encoder\n)'

In [79]:
import joblib
import os

os.makedirs("backend/model", exist_ok=True)

# Priority
joblib.dump(priority_pipeline, "backend/model/priority_pipeline.pkl")
joblib.dump(priority_encoder, "backend/model/priority_encoder.pkl")

# Category
joblib.dump(category_pipeline, "backend/model/category_pipeline.pkl")
joblib.dump(category_encoder, "backend/model/category_encoder.pkl")

['backend/model/category_encoder.pkl']